In [0]:
# R4 task 3 — dq_check
# Runs after the pipeline refreshes bronze + silver and publishes ONE integer to the job:
#     dq_problems = number of FAILED checks (0 = healthy)
# The dq_gate task (If/else condition) reads it as:
#     {{tasks.dq_check.values.dq_problems}} == 0   -> yes-branch -> publish_gold
#
# Check 1 — growth: bronze.claims grew by exactly the expected number of rows since the
#           last run (tracked in <target catalog>.gov.dq_state). Catches missing AND double-processed batches.
# Check 2 — duplicates: no (claim_id, submitted_ts) pair ingested more than once.
#           (Legitimate resubmissions carry a NEW submitted_ts, so they do not fail this.)
#
# NOTE: <target catalog>.gov.dq_state is updated on EVERY run, even if the gate fails afterwards —
#       the next batch compares against the latest bronze count either way.

dbutils.widgets.text("catalog", "claims")   # target catalog (bundle job passes dev/prod catalog)
dbutils.widgets.text("expected_new_rows", "120")
catalog = dbutils.widgets.get("catalog")
expected = int(dbutils.widgets.get("expected_new_rows"))

# gov schema holds dq_state; create it on first run in fresh dev/prod catalogs
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.gov")

# Previous bronze count recorded by the last run (None on the very first run)
prev = None
try:
    row = spark.sql(f"SELECT last_bronze_count FROM {catalog}.gov.dq_state LIMIT 1").collect()
    if row:
        prev = int(row[0]["last_bronze_count"])
except Exception:
    pass  # state table does not exist yet — first ever run

cur = int(spark.sql(f"SELECT COUNT(*) AS c FROM {catalog}.bronze.claims").first()["c"])
growth = (cur - prev) if prev is not None else None

problems = 0

# Check 1: growth equals expected (skipped on the very first run — nothing to compare yet)
growth_ok = (growth == expected) if growth is not None else True
if not growth_ok:
    problems += 1

# Check 2: no duplicate (claim_id, submitted_ts) pairs in bronze

dup_rows = spark.sql(f"""
  SELECT COALESCE(SUM(n - 1), 0) AS extra
  FROM (
    SELECT claim_id, submitted_ts, COUNT(*) AS n
    FROM {catalog}.bronze.claims
    WHERE claim_id IS NOT NULL AND submitted_ts IS NOT NULL
    GROUP BY claim_id, submitted_ts
    HAVING COUNT(*) > 1
  )
""").first()["extra"]
if dup_rows > 0:
    problems += 1

print(f"bronze rows: {cur} (previous: {prev}, growth: {growth}, expected: {expected})")
print(f"duplicate (claim_id, submitted_ts) rows: {dup_rows}")
print(f"checks failed: {problems}")

# Publish to the job — dq_gate reads dq_problems
dbutils.jobs.taskValues.set(key="dq_problems", value=int(problems))
dbutils.jobs.taskValues.set(key="bronze_rows", value=cur)
dbutils.jobs.taskValues.set(key="new_rows", value=growth if growth is not None else -1)

# Record state for the next run
spark.sql(f"""
  CREATE OR REPLACE TABLE {catalog}.gov.dq_state
  COMMENT 'R4: last-seen bronze.claims row count, written by dq_check after each claims_daily run'
  AS SELECT {cur} AS last_bronze_count, current_timestamp() AS updated_at
""")